# 08 · Capstone

Small Barkeep-themed builds that combine everything: a mood model, a memory store, a streaming generator, a registry decorator, and a full token vocabulary. These are miniatures of components you'll build for real later in the project.

### 8.1 — A mood model (composition + property)

Mara's mood is four floats in [0, 1]: warmth, directness, energy, concern. Build a `Mood` class 
that stores them, clamps any out-of-range value to [0, 1] on construction, and exposes a 
`descriptor` property returning `"open"` if warmth > 0.5 else `"reserved"`.

This is a small version of the mood model you'll build in Week 7.

In [1]:
class Mood:
    def __init__(self, warmth, directness, energy, concern):
        self.warmth = max(0, min(1, warmth))
        self.directness = max(0, min(1, directness))
        self.energy = max(0, min(1, energy))
        self.concern = max(0, min(1, concern))

    @property
    def descriptor(self):
        return "open" if self.warmth > 0.5 else "reserved"

# Tests
m = Mood(0.8, 0.5, 0.3, 0.2)
assert m.warmth == 0.8
assert m.descriptor == "open"
clamped = Mood(1.5, -0.2, 0.5, 0.5)
assert clamped.warmth == 1.0       # clamped down
assert clamped.directness == 0.0   # clamped up
assert clamped.descriptor == "open"
print("8.1 passed")

8.1 passed


### 8.2 — A memory store (dunders + custom exception)

A tiny version of Mara's long-term memory. Build `Memory` that stores key→fact pairs. Support 
`memory[key] = fact` (`__setitem__`), `memory[key]` (`__getitem__`, raising a custom 
`UnknownFactError` if missing), `len(memory)`, and `key in memory` (`__contains__`).

In [2]:
class UnknownFactError(Exception):
    pass

class Memory:
    def __init__(self):
        self.facts = {}

    def __setitem__(self, key, value):
        self.facts[key] = value

    def __getitem__(self, key):
        if key not in self.facts:
            raise UnknownFactError(key)
        return self.facts[key]

    def __len__(self):
        return len(self.facts)

    def __contains__(self, key):
        return key in self.facts

# Tests
m = Memory()
m["name"] = "the regular"
m["drink"] = "whiskey, neat"
assert m["name"] == "the regular"
assert len(m) == 2
assert "drink" in m
assert "address" not in m
caught = False
try:
    _ = m["address"]
except UnknownFactError:
    caught = True
assert caught
print("8.2 passed")

8.2 passed


### 8.3 — A streaming generator (generators + state)

Mara's responses stream token by token. Write `stream(text)` — a generator that yields the text one 
word at a time, each prefixed with the words already spoken count is NOT needed; just yield each word. 
Then write `assemble(gen)` that joins a generator of words back into a sentence with single spaces.

This is the lazy-production pattern behind token streaming in Week 9.

In [4]:
def stream(text):
    yield from text.split()

def assemble(gen):
    return " ".join(gen)

# Tests
words = list(stream("snow is still falling"))
assert words == ["snow", "is", "still", "falling"]
assert assemble(stream("the bar is quiet tonight")) == "the bar is quiet tonight"
assert list(stream("")) == []
print("8.3 passed")

8.3 passed


### 8.4 — A registry decorator (decorators + closures)

A common real pattern: a decorator that **registers** functions in a lookup table as a side effect, 
so you can dispatch by name later. You'll use this shape for swappable components (attention types, 
samplers) in Act 1.

Build `registry` (a dict) and a `register(name)` decorator that adds the decorated function to 
`registry` under `name` and returns it unchanged.

In [5]:
registry = {}

def register(name):
    def decorator(func):
        registry[name] = func
        return func
    return decorator

@register("greet")
def greet():
    return "Evening."

@register("farewell")
def farewell():
    return "Mind how you go."


# Tests
assert set(registry.keys()) == {"greet", "farewell"}
assert registry["greet"]() == "Evening."
assert registry["farewell"]() == "Mind how you go."
# functions still usable directly (register returns them unchanged)
assert greet() == "Evening."
print("8.4 passed")

8.4 passed


### 8.5 — Putting it together: a token vocabulary with full protocol

Synthesis. Build `TokenVocab` combining what you've practiced: a bidirectional mapping (from your 
earlier `Vocab`), plus `__len__`, `__contains__`, and `__getitem__` that accepts **either** a string 
(returns its id) **or** an int (returns its token) — dispatching on type. Raise `KeyError` for 
unknowns. `add(token)` returns the id, assigning a new one only if new.

In [9]:
class TokenVocab:
    def __init__(self):
        self.str_to_id = {}
        self.id_to_str = {}

    def add(self, token):
        if token in self.str_to_id:
            return self.str_to_id[token]
        new_id = len(self.str_to_id)
        self.str_to_id[token] =  new_id
        self.id_to_str[new_id] = token
        return new_id

    def __getitem__(self, key):
        if isinstance(key, str):
            if key in self.str_to_id:
                return self.str_to_id[key]
            raise KeyError(key)
        elif isinstance(key, int):
            if key in self.id_to_str:
                return self.id_to_str[key]
            raise KeyError(key)
        raise TypeError(key)


    def __len__(self):
        return len(self.str_to_id)

    def __contains__(self, token):
        return token in self.str_to_id

# Tests
v = TokenVocab()
assert v.add("hello") == 0
assert v.add("world") == 1
assert v.add("hello") == 0          # existing
assert v["world"] == 1              # str -> id
assert v[0] == "hello"              # int -> token
assert len(v) == 2
assert "hello" in v
assert "missing" not in v
for bad in ["nope", 99]:
    caught = False
    try:
        _ = v[bad]
    except KeyError:
        caught = True
    assert caught
print("8.5 passed")

8.5 passed
